In [0]:
%run ../../config/metadata_setup

In [0]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType, DecimalType
from pyspark.sql.functions import col, lit, filter, to_date
import os

In [0]:
class OrderDetailsSilverStreaming:
    def __init__(self, spark):
        self.spark = spark
        self.catalog_name = CONFIG['catalog']['name']
        self.schema_name = CONFIG['catalog']['schema']

        # Paths
        self.schema_path = CONFIG['paths']['schemas']
        self.checkpoint_path = CONFIG['paths']['checkpoints']

        # up-stream table name
        self.upstream_table_name = TABLES['bronze']['order_details']

        # down-stream table name
        self.downstream_table_name = TABLES['silver']['order_details']
    
    # --------------------------------
    # Read bronze streaming data
    # --------------------------------
    def read_stream(self):
        return (
            spark.readStream.table(f"{self.catalog_name}.{self.schema_name}.{self.upstream_table_name}")
        )

    # ---------------------------
    # Change data types
    # ISO 8601 Date Format
    # ---------------------------
    def change_data_types(self, df):
        df_casted = (
            df.withColumn('amount', col('amount').cast(DecimalType(10, 2)))
            .withColumn("profit", col('profit').cast(DecimalType(10, 2)))
            .withColumn("quantity",  col('profit').cast(IntegerType()))
        )
        return df_casted

    # --------------------------------
    # Columns to keep
    # --------------------------------
    def columns_to_keep(self, df):
        df = df.select('surrogate_key','order_id', 'amount', 'profit', 'quantity', 'category','sub_category', '_rescued_data')
        return df
    
    # ----------------------
    # Upsert to silver
    # ----------------------
    def upsert_to_silver(self, microBatchDF, batchId):
        # Deduplicate witinn the micro-batch
        dedup_df = microBatchDF.dropDuplicates(['surrogate_key'])
        # Create temp view
        dedup_df.createOrReplaceTempView("v_deduped_orders")

        # Use local variables to avoid capturing self
        catalog = self.catalog_name
        schema = self.schema_name
        table = self.downstream_table_name

        # Merge inot silver table - use microBatchDF.sparkSession instead of self.spark
        microBatchDF.sparkSession.sql(f"""
            MERGE INTO {catalog}.{schema}.{table} AS t
            USING deduped_orders AS s
            ON t.surrogate_key = s.surrogate_key
            WHEN NOT MATCHED THEN 
            INSERT *                          
            """)

    # -------------------------
    # Display upstream data
    # -------------------------
    def display_upstream_data(self):
        return self.spark.table(f"{self.catalog_name}.{self.schema_name}.{self.upstream_table_name}")

    # --------------------------------
    # Only for droping table
    #---------------------------------
    #def drop_table(self):
    #    dbutils.fs.rm(f"{self.checkpoint_path}/_{self.downstream_table_name}", True)

    # ---------------------------------
    # write to delta table (streaming)
    # ---------------------------------
    def write_stream(self, df):
        try:
            query =(
                df.writeStream
                    .option("checkpointLocation", f"{self.checkpoint_path}/_{self.downstream_table_name}")
                    .foreachBatch(self.upsert_to_silver)
                    .trigger(availableNow=True)
                    .option("mergeSchema", "true")
                    .table(f"{self.catalog_name}.{self.schema_name}.{self.downstream_table_name}")
            )
            query.awaitTermination()
        except Exception as e:
            print(e)

    def run(self):
        df = self.read_stream()
        df = self.columns_to_keep(df)
        df_casted = self.change_data_types(df)
        self.write_stream(df_casted)
        print("successfull run")

In [0]:
obj  = OrderDetailsSilverStreaming(spark)
obj.run()